# Spectral Plots That Actually Show the Chemistry
### 2.5 — Plotting That Reveals Chemistry

A plot is not decoration you add at the end of an analysis. **A good plot is an
analytical instrument** — it is how you, and everyone who reads your work, turn
a wall of numbers into a chemical conclusion. Overlaying spectra exposes a
batch that drifted. Shading a band marks exactly where you integrated. A
reversed axis silently tells a spectroscopist "this is IR."

The default `matplotlib` plot answers one question: *"what does the data look
like?"* That is rarely the question you actually have. Your questions are
chemical: *Is the analyte band growing with concentration? Did this batch pick
up a contaminant? Is that bump real or is it noise?*

This notebook is **not a matplotlib tutorial.** `matplotlib` is just the pen.
The skill we are practising is **communicating chemical information** — making a
figure that lets a reader reach the right conclusion, and refusing to make one
that leads them to the wrong one.

The pattern in every section is the same:

> **the default plot → the revised plot → the chemical question it now answers.**


## 1. Setup: import the tools and build our cast of spectra

We will reuse three simulated datasets throughout. Because they are *simulated*,
we know the true chemistry behind every line — so when a plot reveals (or hides)
something, we can be certain whether the plot is telling the truth.

In [ ]:
# The standard scientific Python stack -- nothing exotic.
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# Our project's UV-Vis simulator. It returns a `Dataset` carrying a shared
# wavelength axis (`x`), a 2D data matrix (`X`), and -- crucially for plotting --
# the axis labels and units, so our figures can be labelled correctly *from the
# data itself* instead of from hand-typed strings that drift out of date.
#
# Installed once from the repo root with:  pip install -e ".[notebooks]"
from simulated_data import uvvis

# A folder for the publication-quality figures we export later (git-ignored).
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

np.set_printoptions(precision=3, suppress=True)
print("Ready.")

**Dataset 1 — one clean spectrum** (`clean`). Two well-separated absorbance
bands near 450 nm and 600 nm on a gentle instrumental baseline. This is our
reference figure.

In [ ]:
# `seed` makes the spectrum identical on every run, so your figures match the
# committed ones exactly. `baseline=0.12` adds a gentle instrumental bow.
clean = uvvis.simulate(seed=7, baseline=0.12)

wl = clean.x              # shared wavelength axis, shape (n_points,)
absorbance = clean.single()   # the lone spectrum as a 1D array

print(f"axis : {clean.x_label} ({clean.x_unit})")
print(f"signal: {clean.y_label}")
print(f"points: {wl.size}   span: {wl.min():.0f}-{wl.max():.0f} {clean.x_unit}")

**Dataset 2 — a Beer-Lambert concentration series** (`series`). The same
analyte measured at five concentrations. By Beer's law the band *heights* scale
with concentration while the baseline (an instrument property) does not. This is
the natural case for an **overlay** with a **sequential** colour scheme.

In [ ]:
concs = [0.25, 0.50, 1.00, 1.50, 2.00]   # relative concentrations
series = uvvis.simulate(
    concentration=concs,   # one spectrum per concentration ...
    n_samples=len(concs),  # ... so n_samples must match the list length
    baseline=0.10,
    noise=0.004,
    seed=3,
)
print("series shape:", series.X.shape)
print("true concentrations:", list(series.meta["concentration"]))

**Dataset 3 — three production batches** (`batches`). Same nominal product,
but **batch C has picked up a contaminant** that absorbs near 520 nm — right in
the valley between the two main bands. This is the kind of difference that a
single spectrum hides and a good overlay exposes. We build it from explicit
`(center, FWHM, amplitude)` peak lists so we know the ground truth.

In [ ]:
# (center_nm, FWHM_nm, amplitude_AU)
spec_clean   = [(450, 40, 0.80), (600, 60, 0.40)]                 # on-spec product
spec_contam  = [(450, 40, 0.80), (520, 30, 0.30), (600, 60, 0.40)]  # + contaminant

# Same baseline + noise level for all three, so any difference we see is REAL
# chemistry, not a plotting artefact. Different seeds = independent noise draws.
batch_A = uvvis.simulate(peaks=spec_clean,  baseline=0.10, noise=0.005, seed=11)
batch_B = uvvis.simulate(peaks=spec_clean,  baseline=0.10, noise=0.005, seed=12)
batch_C = uvvis.simulate(peaks=spec_contam, baseline=0.10, noise=0.005, seed=13)

batches = {"Batch A": batch_A, "Batch B": batch_B, "Batch C": batch_C}
print("Built", len(batches), "batches on a shared", clean.x_unit, "axis.")

## 2. The default plot, and what it hides

Here is the plot most people make first: hand the numbers to `matplotlib` and
move on.

In [ ]:
# The "just show me the data" plot.
plt.plot(series.x, series.X.T)
plt.show()

It is not *wrong* — but ask what a colleague could actually conclude from it:

- **What technique is this?** No way to tell.
- **What are the axes?** No labels, no units. The x-numbers could be nanometres,
  wavenumbers, minutes, or m/z.
- **Which line is which sample?** The five concentrations are five
  indistinguishable default colours with no legend.
- **Where is the chemistry?** Nothing points to the analyte bands.

Every fix from here on is about answering one of those questions. None of them
is about `matplotlib` for its own sake.

## 3. Axis labels, units, and direction

**An unlabelled axis is not a measurement.** A number with no unit cannot be
checked, reproduced, or trusted. The single highest-value habit in scientific
plotting is to label both axes with quantity *and* unit, every time.

We pull the labels straight from the `Dataset` so they can never drift out of
sync with the data.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(clean.x, clean.single(), color="#0072B2")

# Labels + units come FROM THE DATA, not from memory. If the simulator changes
# units tomorrow, this figure stays correct with no edits.
ax.set_xlabel(f"{clean.x_label} ({clean.x_unit})")
ax.set_ylabel(clean.y_label)
ax.set_title("UV-Vis absorbance of the reference sample")

plt.tight_layout()
plt.show()

That is already a figure you could hand to someone: they know it is a UV-Vis
absorbance spectrum in nm, with two bands.

**A note on axis direction.** UV-Vis is plotted left-to-right in increasing
wavelength, as above. But convention is chemistry-specific and carries meaning:
infrared and Raman spectra are almost always plotted with **wavenumber
decreasing left-to-right** (high cm$^{-1}$ on the left). Plotting an IR spectrum
the "normal" way will read as wrong to any spectroscopist. You reverse an axis
with `ax.invert_xaxis()` — the convention *is* part of communicating clearly to
your audience.

## 4. Colour that carries meaning

`matplotlib`'s default colour cycle assigns colours in the order lines are drawn.
Those colours mean **nothing** — they are just "first, second, third." We can do
far better by letting colour *encode a variable*:

- **Ordered quantity (concentration, time, temperature):** use a **sequential**
  colormap so light-to-dark maps to low-to-high. The reader sees the trend in
  the colour, not just the line positions.
- **Distinct categories (samples, classes, batches):** use a **categorical,
  colourblind-safe** palette of well-separated hues.

Two things to avoid: the **rainbow/`jet`** colormap (its bright yellow band
creates fake "features" where the data is smooth — more on this in Section 10),
and **red/green** as your only contrast (the most common form of colour-vision
deficiency makes them indistinguishable).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))

# A sequential colormap mapped to concentration: low = light, high = dark.
cmin, cmax = min(concs), max(concs)
norm = Normalize(vmin=cmin, vmax=cmax)
cmap = plt.cm.viridis

for i, c in enumerate(series.meta["concentration"]):
    ax.plot(series.x, series.X[i], color=cmap(norm(c)), lw=1.8)

ax.set_xlabel(f"{series.x_label} ({series.x_unit})")
ax.set_ylabel(series.y_label)
ax.set_title("UV-Vis concentration series (colour = concentration)")

# A colourbar is the legend for a continuous variable -- far cleaner than five
# legend entries, and it shows the scale.
sm = ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Relative concentration")

plt.tight_layout()
plt.show()

Now the figure *tells the Beer-Lambert story by itself*: as concentration
rises (colour darkens), both bands grow in proportion while the baseline stays
put. The colour is doing analytical work.

For **categorical** data, here is the colourblind-safe palette we use for the
three batches (the Okabe-Ito set — distinguishable across all common types of
colour-vision deficiency).

In [ ]:
# Okabe-Ito colourblind-safe categorical palette.
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#CC79A7",
             "#56B4E9", "#D55E00", "#F0E442", "#000000"]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for (name, ds), color in zip(batches.items(), OKABE_ITO):
    ax.plot(ds.x, ds.single(), label=name, color=color, lw=1.6)

ax.set_xlabel(f"{batch_A.x_label} ({batch_A.x_unit})")
ax.set_ylabel(batch_A.y_label)
ax.set_title("Three batches (colour = category)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Overlays that expose differences

The overlay you just drew is the single most useful comparison in spectroscopy.
Look again at **Batch C** in the plot above: there is a clear extra band near
520 nm, in the valley between the two main peaks. If you had plotted each batch
on its own axes you might never have noticed — your eye has nothing to compare
against. **The overlay turns three separate measurements into one comparison,
and the comparison is where the chemistry lives.**

Let us make that contaminant impossible to miss by zooming the overlay onto the
valley region.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for (name, ds), color in zip(batches.items(), OKABE_ITO):
    ax.plot(ds.x, ds.single(), label=name, color=color, lw=1.8)

ax.set_xlim(480, 580)          # zoom onto the valley between the main bands
ax.set_ylim(0, 0.6)            # honest y-axis that still starts at 0 (see Sec 10)
ax.set_xlabel(f"{batch_A.x_label} ({batch_A.x_unit})")
ax.set_ylabel(batch_A.y_label)
ax.set_title("Same data, zoomed to the valley: Batch C has an extra band")
ax.legend()
plt.tight_layout()
plt.show()

**The one rule for honest overlays: only overlay comparable spectra.** Here all
three batches were measured the same way (same baseline, same noise level), so
the comparison is fair and the 520 nm band is genuinely chemistry. If one
spectrum had been baseline-corrected and another not, or one normalised and
another not, the overlay would invite a false conclusion. Overlay apples with
apples.

## 6. Highlighting regions of interest

Often the figure needs to say *"look here, and here is why."* Shading a band with
`axvspan` marks an integration window, a diagnostic region, or the exact place a
contaminant shows up — and a one-line caption gives the **chemical reason** for
that window.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(clean.x, clean.single(), color="#333333", lw=1.6)

# Shade the two analyte bands we'd integrate, and the contaminant window.
ax.axvspan(420, 480, color="#0072B2", alpha=0.15, label="Band 1 window (~450 nm)")
ax.axvspan(560, 640, color="#009E73", alpha=0.15, label="Band 2 window (~600 nm)")
ax.axvspan(505, 535, color="#D55E00", alpha=0.18, label="Contaminant watch (~520 nm)")

ax.set_xlabel(f"{clean.x_label} ({clean.x_unit})")
ax.set_ylabel(clean.y_label)
ax.set_title("Integration windows and a contaminant-watch region")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

The shading does two scientific jobs at once: it documents **exactly** which
wavelengths you integrated (reproducibility), and it directs a reader's eye to
the region you want them to judge. A shaded band is a far stronger statement than
"we integrated around 450 nm" buried in the text.

## 7. Annotations that point at the chemistry

Annotations should *name the chemistry*, not decorate the plot. Label the
analyte bands, mark $\lambda_{max}$, and call out the contaminant — so the figure
is self-explanatory even pulled out of context into a slide.

In [ ]:
spec = batch_C.single()   # the batch WITH the contaminant
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(batch_C.x, spec, color="#333333", lw=1.6)

# Find lambda_max of the main 450 nm band programmatically, then annotate it.
mask = (batch_C.x > 420) & (batch_C.x < 480)
i_peak = np.argmax(np.where(mask, spec, -np.inf))
lam_max = batch_C.x[i_peak]
ax.annotate(
    f"$\\lambda_{{max}}$ = {lam_max:.0f} nm",
    xy=(lam_max, spec[i_peak]),
    xytext=(lam_max - 5, spec[i_peak] + 0.12),
    arrowprops=dict(arrowstyle="->", color="#0072B2"),
    color="#0072B2", fontweight="bold",
)

# Call out the contaminant band with an arrow from above.
ax.annotate(
    "contaminant\n(~520 nm)",
    xy=(520, spec[np.argmin(np.abs(batch_C.x - 520))]),
    xytext=(540, 0.45),
    arrowprops=dict(arrowstyle="->", color="#D55E00"),
    color="#D55E00", ha="left",
)

# A plain text label for the second band -- no arrow needed.
ax.text(600, spec[np.argmin(np.abs(batch_C.x - 600))] + 0.03,
        "Band 2", ha="center", color="#009E73")

ax.set_xlabel(f"{batch_C.x_label} ({batch_C.x_unit})")
ax.set_ylabel(batch_C.y_label)
ax.set_title("Annotated spectrum: every feature named")
plt.tight_layout()
plt.show()

Note we found $\lambda_{max}$ **from the data** with `np.argmax`, not by typing a
number. Annotations computed from the array stay correct if the data changes —
and they double as a tiny sanity check that the peak is where you think it is.

## 8. Publication-quality formatting

"Publication quality" is mostly a handful of consistent defaults: readable font
sizes, a sensible figure size and resolution, ticks that point the right way,
no chartjunk, and a clean frame. Set them once via `rcParams` and every figure
afterwards inherits them.

In [ ]:
# A compact, reusable style. Apply once; all later figures pick it up.
plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,            # crisp on screen
    "savefig.dpi": 300,           # print/journal resolution on export
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "axes.spines.top": False,     # drop the top/right frame ("despine")
    "axes.spines.right": False,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "legend.frameon": False,
    "lines.linewidth": 1.8,
})
print("Publication style applied.")

In [ ]:
# A single paper-ready figure that uses everything from this notebook:
# meaningful colour, labelled axes, a shaded ROI, and a computed annotation.
fig, ax = plt.subplots()
cmap, norm = plt.cm.viridis, Normalize(min(concs), max(concs))
for i, c in enumerate(series.meta["concentration"]):
    ax.plot(series.x, series.X[i], color=cmap(norm(c)))

ax.axvspan(420, 480, color="0.85", alpha=0.6, zorder=0)  # integration window
ax.set_xlabel(f"{series.x_label} ({series.x_unit})")
ax.set_ylabel(series.y_label)
ax.set_title("UV-Vis calibration series with integration window")

sm = ScalarMappable(norm=norm, cmap=cmap)
fig.colorbar(sm, ax=ax).set_label("Relative concentration")

fig.tight_layout()

# Export at 300 dpi as PNG (slides) and SVG (vector, for journals/posters).
fig.savefig(EXPORTS / "calibration_series.png", bbox_inches="tight")
fig.savefig(EXPORTS / "calibration_series.svg", bbox_inches="tight")
plt.show()
print("Saved PNG + SVG to", EXPORTS.resolve())

## 9. Avoiding misleading visualizations

A plot can be technically correct and still lead a reader to a false conclusion.
This is where plotting becomes an integrity issue: **a misleading figure is a
misleading result.** Three of the most common traps, each shown honest-vs-
misleading on identical data.

### Trap 1 — the truncated y-axis

Batches A and B are the same product within noise. Start the y-axis at zero and
that is obvious. Start it at 0.38 and you have manufactured a "difference" out of
noise.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

for ax in (ax1, ax2):
    ax.plot(batch_A.x, batch_A.single(), color="#0072B2", label="Batch A")
    ax.plot(batch_B.x, batch_B.single(), color="#E69F00", label="Batch B")
    ax.set_xlim(430, 470)
    ax.set_xlabel(f"{batch_A.x_label} ({batch_A.x_unit})")
    ax.legend()

ax1.set_ylim(0, 0.95)
ax1.set_title("Honest: y-axis starts at 0\n(A and B agree)")
ax1.set_ylabel(batch_A.y_label)

ax2.set_ylim(0.78, 0.84)
ax2.set_title("Misleading: zoomed y-axis\n(fake 'difference' from noise)")

plt.tight_layout()
plt.show()

Same data, opposite conclusions. Zooming the y-axis is legitimate *when you say
so and there is a real effect to see* — but for intensity comparisons, defaulting
to a zero baseline keeps you honest.

### Trap 2 — the rainbow (`jet`) colormap

`jet` is not perceptually uniform: its bright cyan/yellow bands read as
"features" even where the data is perfectly smooth, and it is hostile to
colourblind readers. A perceptual colormap like `viridis` shows the *true*
structure.

In [ ]:
# Build a smooth 2D map: the concentration series stacked as an image.
M = series.X

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
im1 = ax1.imshow(M, aspect="auto", cmap="jet",
                 extent=[series.x.min(), series.x.max(), 0, len(concs)])
ax1.set_title("Misleading: 'jet' invents banding")
ax1.set_xlabel(f"{series.x_label} ({series.x_unit})")
ax1.set_ylabel("sample #")
fig.colorbar(im1, ax=ax1)

im2 = ax2.imshow(M, aspect="auto", cmap="viridis",
                 extent=[series.x.min(), series.x.max(), 0, len(concs)])
ax2.set_title("Honest: 'viridis' shows true structure")
ax2.set_xlabel(f"{series.x_label} ({series.x_unit})")
ax2.set_ylabel("sample #")
fig.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

### Trap 3 — overlaying differently-processed spectra

If one spectrum is baseline-corrected and another is not, an overlay makes them
look chemically different when the only difference is processing. Below, the two
spectra are the *same chemistry*; only the baseline treatment differs — yet the
overlay screams "different sample."

Other traps worth naming (not plotted here): **dual y-axes** that imply a
correlation between unrelated quantities by lining up their wiggles, and
**over-smoothing** a noisy spectrum so a figure hides the noise it was built on
(we cover honest smoothing in notebook 3.2).

In [ ]:
raw = batch_A.single()
# Same spectrum, but subtract a crude baseline from ONE of them.
baseline_guess = np.linspace(raw[0], raw[-1], raw.size)
corrected = raw - baseline_guess

fig, ax = plt.subplots()
ax.plot(batch_A.x, raw, color="#0072B2", label="Batch A (raw)")
ax.plot(batch_A.x, corrected, color="#D55E00", label="Batch A (baseline-subtracted)")
ax.set_xlabel(f"{batch_A.x_label} ({batch_A.x_unit})")
ax.set_ylabel(batch_A.y_label)
ax.set_title("Same chemistry, different processing -- NOT a fair overlay")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Bottling it: a reusable `plot_spectra` helper

Every good habit above — labels from the data, meaningful colour, a clean frame —
can be wrapped into one function you carry into every later notebook. This is the
"plotting vocabulary" the rest of the series reuses.

In [ ]:
def plot_spectra(ds, *, labels=None, color_by=None, cmap="viridis",
                 highlight=None, ax=None, title=None):
    '''Plot the spectra in a Dataset with sane scientific defaults.

    Parameters
    ----------
    ds : Dataset
        Any dataset from `simulated_data` (carries axis, X, and labels/units).
    labels : list of str, optional
        One legend label per spectrum (categorical case).
    color_by : array-like, optional
        A per-spectrum quantity (e.g. concentration). If given, spectra are
        coloured by a sequential colormap + colourbar instead of a legend.
    cmap : str
        Colormap name for the `color_by` case.
    highlight : tuple (lo, hi), optional
        A wavelength window to shade as a region of interest.
    ax : matplotlib Axes, optional
        Draw into an existing axis; otherwise a new figure is made.
    title : str, optional
        Plot title.

    Returns
    -------
    matplotlib Axes
    '''
    if ax is None:
        _, ax = plt.subplots()

    if color_by is not None:
        color_by = np.asarray(color_by, dtype=float)
        norm = Normalize(color_by.min(), color_by.max())
        cm = plt.get_cmap(cmap)
        for row, c in zip(ds.X, color_by):
            ax.plot(ds.x, row, color=cm(norm(c)))
        sm = ScalarMappable(norm=norm, cmap=cm)
        ax.figure.colorbar(sm, ax=ax)
    else:
        for i, row in enumerate(ds.X):
            lab = labels[i] if labels is not None else None
            ax.plot(ds.x, row, label=lab, color=OKABE_ITO[i % len(OKABE_ITO)])
        if labels is not None:
            ax.legend()

    if highlight is not None:
        ax.axvspan(*highlight, color="0.85", alpha=0.6, zorder=0)

    # Labels + units always come from the Dataset -- never wrong, never stale.
    ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
    ax.set_ylabel(ds.y_label)
    if title:
        ax.set_title(title)
    return ax


# One-liner reproductions of earlier figures, proving the helper works:
plot_spectra(series, color_by=series.meta["concentration"],
             highlight=(420, 480), title="Calibration series (via helper)")
plt.tight_layout(); plt.show()

In [ ]:
# And the categorical batch overlay, also in one line:
plot_spectra(batch_A.__class__(  # stack the three batches into one Dataset
                 x=batch_A.x,
                 X=np.vstack([b.single() for b in batches.values()]),
                 y=None,
                 meta=batch_A.meta.iloc[[0, 0, 0]].reset_index(drop=True),
                 x_label=batch_A.x_label, x_unit=batch_A.x_unit,
                 y_label=batch_A.y_label),
             labels=list(batches.keys()),
             title="Three batches (via helper)")
plt.tight_layout(); plt.show()

## 11. What this means scientifically

The figure is **part of the argument**, not an afterthought to it. Three ideas to
carry into every plot you make:

1. **Start from the chemical question.** Before choosing a chart, name what a
   reader should be able to conclude — "the analyte grows linearly," "batch C is
   contaminated," "the bump is noise." Then build the plot that answers it.
2. **Let the visual channels do work.** Colour can encode concentration; shading
   can mark an integration window; an annotation can name $\lambda_{max}$. A plot
   that uses its channels deliberately needs less caption.
3. **Honesty is a design constraint.** A truncated axis, a rainbow colormap, or
   an unfair overlay can manufacture a conclusion the data does not support. The
   same skill that *reveals* chemistry can *fabricate* it — so the rules in
   Section 9 are not style preferences, they are part of doing the science right.

A good spectral plot, in the end, is just a measurement you can see.

## Key Takeaways

- **The figure is part of the argument.** Start from the chemical question — what
  should a reader be able to conclude? — then build the plot that answers it.
- **Let the visual channels do work:** colour can encode concentration, shading
  can mark an integration window, an annotation can name $\lambda_{max}$.
- **Honesty is a design constraint.** The same skills that *reveal* chemistry can
  *fabricate* it, so the fairness rules are part of doing the science right.
- **One repeatable loop upgrades any spectral plot:** *Label* → *Colour with
  meaning* → *Overlay* comparable spectra → *Mark* the chemistry → *Check
  honesty*.

## Practical Checklist

- [ ] Axis labels carry quantity **and** unit (pull them from the `Dataset`).
- [ ] Sequential colormap for ordered quantities; categorical + colourblind-safe
  palette for distinct groups.
- [ ] Overlay only **comparable** (identically-processed) spectra.
- [ ] Shade regions of interest; annotate key features.
- [ ] Intensity axes start at a sensible zero; use a perceptual colormap
  (`viridis`), never `jet`.

## Common Mistakes

- A truncated y-axis that manufactures a "failed" batch out of identical data.
- A rainbow / `jet` colormap inventing banding that isn't in the data.
- Overlaying differently-processed spectra (one baseline-corrected, one not).
- Dual y-axes that imply a correlation between unrelated quantities.
- Over-smoothing a noisy spectrum so the figure hides the noise it was built on
  (we cover honest smoothing in 3.2).

## Reporting Guidance

- State any processing (baseline, smoothing, normalization) in the caption, so an
  overlay can be read fairly.
- Export both a raster (PNG, ≥300 dpi) for documents and a vector (SVG) for
  publication.

## Next Lesson

**2.6 — Missing Values and Detector Dropouts.** We now have spectra worth
plotting; next we deal with the gaps the detector leaves behind — dropouts,
saturation, and dead channels — and decide honestly whether to drop, interpolate,
or flag them.